In [8]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "torch-geometric", "scikit-learn", "-q"], check=False)
import torch_geometric
print(f"torch-geometric: {torch_geometric.__version__}")


torch-geometric: 2.8.0.post1


# GNN Model Training


## 1. Load Split Data Tensors


In [9]:
import os, glob
import torch
from torch_geometric.data import HeteroData

LOCAL_PATH = '/home/shreyas-nalle/Desktop/Delusional/model/split_data.pt'
paths_to_check = [LOCAL_PATH, 'model/split_data.pt', 'split_data.pt']
kaggle_glob = glob.glob('/kaggle/input/**/split_data*', recursive=True)
if kaggle_glob: paths_to_check.insert(0, kaggle_glob[0])

split_path = next((p for p in paths_to_check if os.path.exists(p)), None)
if split_path is None: raise FileNotFoundError('split_data.pt not found!')
split = torch.load(split_path, weights_only=False)

def to_hetero_data(data):
    h = HeteroData()
    h['node'].x = data.x
    h['node', 'to', 'node'].edge_index = data.edge_index
    h['node', 'rev_to', 'node'].edge_index = data.edge_index.flipud()
    h['node', 'to', 'node'].edge_attr = data.edge_attr
    h['node', 'rev_to', 'node'].edge_attr = data.edge_attr.clone()
    if h['node', 'rev_to', 'node'].edge_attr.shape[1] >= 8:
        h['node', 'rev_to', 'node'].edge_attr[:, [-4, -3]] = h['node', 'rev_to', 'node'].edge_attr[:, [-3, -4]]
    h['node', 'to', 'node'].y = data.y
    return h

tr_data = to_hetero_data(split['tr_data'])
val_data = to_hetero_data(split['val_data'])
te_data = to_hetero_data(split['te_data'])
tr_inds, val_inds, te_inds = split['tr_inds'], split['val_inds'], split['te_inds']
print(f'Loaded split_data from {split_path}: Train={tr_inds.shape[0]:,}, Val={val_inds.shape[0]:,}, Test={te_inds.shape[0]:,}')


Loaded split_data from /kaggle/input/models/shreyasnalle/split-data/pytorch/default/1/split_data.pt: Train=4,466,821, Val=530,666, Test=2,513


## 2. Add Unique Edge IDs


In [10]:
def add_arange_ids(data_list):
    for data in data_list:
        n_edges = data['node', 'to', 'node'].edge_attr.shape[0]
        ids = torch.arange(n_edges).view(-1, 1)
        data['node', 'to', 'node'].edge_attr = torch.cat([ids, data['node', 'to', 'node'].edge_attr], dim=1)
        data['node', 'rev_to', 'node'].edge_attr = torch.cat([ids.clone(), data['node', 'rev_to', 'node'].edge_attr], dim=1)
add_arange_ids([tr_data, val_data, te_data])
print(f"Edge attribute features shape with ID: {tr_data['node', 'to', 'node'].edge_attr.shape}")


Edge attribute features shape with ID: torch.Size([4466821, 5])


## 3. Create Mini-Batch DataLoaders


In [11]:
import torch
from torch_geometric.data import HeteroData

BATCH_SIZE = 8192
NUM_NEIGHBORS = 50

class HeteroEdgeLoader:
    def __init__(self, data, edge_inds, batch_size, shuffle=False, num_neighbors=50, add_ego_ids=True, balanced=False):
        self.data = data
        self.edge_inds = edge_inds
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_neighbors = num_neighbors
        self.add_ego_ids = add_ego_ids
        self.balanced = balanced
        n_nodes = data['node'].x.shape[0]
        
        src_to = data['node', 'to', 'node'].edge_index[0]
        self._perm_to = src_to.argsort()
        counts_to = torch.bincount(src_to, minlength=n_nodes)
        self._ptr_to = torch.cat([torch.zeros(1, dtype=torch.long), counts_to.cumsum(0)])
        
        src_rev = data['node', 'rev_to', 'node'].edge_index[0]
        self._perm_rev = src_rev.argsort()
        counts_rev = torch.bincount(src_rev, minlength=n_nodes)
        self._ptr_rev = torch.cat([torch.zeros(1, dtype=torch.long), counts_rev.cumsum(0)])
        
        if balanced:
            y_edges = data['node', 'to', 'node'].y[edge_inds]
            pos_mask = y_edges == 1
            self._pos_inds = edge_inds[pos_mask]
            self._neg_inds = edge_inds[~pos_mask]
            print(f'HeteroEdgeLoader Balanced Sampling: {len(self._pos_inds):,} positive, {len(self._neg_inds):,} negative edges')

    def __len__(self):
        return (len(self.edge_inds) + self.batch_size - 1) // self.batch_size

    def _neighbor_edges(self, nodes, ptr, perm):
        starts, ends = ptr[nodes], ptr[nodes + 1]
        parts = []
        for s, e in zip(starts.tolist(), ends.tolist()):
            if s == e: continue
            eidx = perm[s:e]
            if len(eidx) > self.num_neighbors:
                eidx = eidx[torch.randperm(len(eidx))[:self.num_neighbors]]
            parts.append(eidx)
        return torch.cat(parts) if parts else torch.empty(0, dtype=torch.long)

    def _make_batch(self, chunk):
        seed_edges = chunk
        seed_nodes = self.data['node', 'to', 'node'].edge_index[:, seed_edges].reshape(-1).unique()
        
        nbr_to = self._neighbor_edges(seed_nodes, self._ptr_to, self._perm_to)
        nbr_rev = self._neighbor_edges(seed_nodes, self._ptr_rev, self._perm_rev)
        
        all_to = torch.cat([seed_edges, nbr_to]).unique()
        all_rev = torch.cat([seed_edges, nbr_rev]).unique()
        
        nodes_to = self.data['node', 'to', 'node'].edge_index[:, all_to].reshape(-1)
        nodes_rev = self.data['node', 'rev_to', 'node'].edge_index[:, all_rev].reshape(-1)
        all_nodes = torch.cat([nodes_to, nodes_rev]).unique()
        
        node_map = torch.full((self.data['node'].x.shape[0],), -1, dtype=torch.long)
        node_map[all_nodes] = torch.arange(len(all_nodes))
        
        sub_ei_to = node_map[self.data['node', 'to', 'node'].edge_index[:, all_to]]
        sub_ei_rev = node_map[self.data['node', 'rev_to', 'node'].edge_index[:, all_rev]]
        
        batch = HeteroData()
        x_batch = self.data['node'].x[all_nodes]
        if self.add_ego_ids:
            ego_feat = torch.zeros((len(all_nodes), 1), dtype=torch.float)
            ego_nodes = self.data['node', 'to', 'node'].edge_index[:, seed_edges].reshape(-1).unique()
            ego_feat[node_map[ego_nodes]] = 1.0
            x_batch = torch.cat([x_batch, ego_feat], dim=1)
        
        batch['node'].x = x_batch
        batch['node', 'to', 'node'].edge_index = sub_ei_to
        batch['node', 'rev_to', 'node'].edge_index = sub_ei_rev
        batch['node', 'to', 'node'].edge_attr = self.data['node', 'to', 'node'].edge_attr[all_to]
        batch['node', 'rev_to', 'node'].edge_attr = self.data['node', 'rev_to', 'node'].edge_attr[all_rev]
        batch['node', 'to', 'node'].y = self.data['node', 'to', 'node'].y[all_to]
        
        batch.input_id = chunk
        batch._seed_ids = self.data['node', 'to', 'node'].edge_attr[seed_edges, 0]
        return batch

    def __iter__(self):
        if self.balanced:
            half = self.batch_size // 2
            n_batches = len(self)
            for _ in range(n_batches):
                pos_i = torch.randint(0, len(self._pos_inds), (half,))
                neg_i = torch.randint(0, len(self._neg_inds), (half,))
                chunk = torch.cat([self._pos_inds[pos_i], self._neg_inds[neg_i]])
                yield self._make_batch(chunk)
        else:
            n = len(self.edge_inds)
            order = torch.randperm(n) if self.shuffle else torch.arange(n)
            for start in range(0, n, self.batch_size):
                chunk = self.edge_inds[order[start:start + self.batch_size]]
                yield self._make_batch(chunk)

tr_loader = HeteroEdgeLoader(tr_data, tr_inds, BATCH_SIZE, shuffle=True, num_neighbors=NUM_NEIGHBORS, add_ego_ids=True, balanced=True)
val_loader = HeteroEdgeLoader(val_data, val_inds, BATCH_SIZE, shuffle=False, num_neighbors=NUM_NEIGHBORS, add_ego_ids=True, balanced=False)
te_loader = HeteroEdgeLoader(te_data, te_inds, BATCH_SIZE, shuffle=False, num_neighbors=NUM_NEIGHBORS, add_ego_ids=True, balanced=False)
sample_batch = next(iter(tr_loader))
print(f'HeteroLoaders initialized: Train={len(tr_loader)}, Val={len(val_loader)}, Test={len(te_loader)}')


HeteroEdgeLoader Balanced Sampling: 1,008 positive, 4,465,813 negative edges
HeteroLoaders initialized: Train=546, Val=65, Test=1


## 4. Define GINe Model Architecture


In [12]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, BatchNorm, Linear, to_hetero

class GINe(torch.nn.Module):
    def __init__(self, num_features, num_gnn_layers, n_classes=2, n_hidden=100, edge_updates=False, edge_dim=None, dropout=0.0, final_dropout=0.5):
        super().__init__()
        self.n_hidden = n_hidden
        self.num_gnn_layers = num_gnn_layers
        self.edge_updates = edge_updates
        self.final_dropout = final_dropout
        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)
        self.convs = nn.ModuleList()
        self.emlps = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        for _ in range(self.num_gnn_layers):
            conv = GINEConv(nn.Sequential(nn.Linear(self.n_hidden, self.n_hidden), nn.ReLU(), nn.Linear(self.n_hidden, self.n_hidden)), edge_dim=self.n_hidden)
            if self.edge_updates:
                self.emlps.append(nn.Sequential(nn.Linear(3 * self.n_hidden, self.n_hidden), nn.ReLU(), nn.Linear(self.n_hidden, self.n_hidden)))
            self.convs.append(conv)
            self.batch_norms.append(BatchNorm(n_hidden))
        self.mlp = nn.Sequential(Linear(n_hidden * 3, 50), nn.ReLU(), nn.Dropout(self.final_dropout), Linear(50, 25), nn.ReLU(), nn.Dropout(self.final_dropout), Linear(25, n_classes))
    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](torch.cat([x[src], x[dst], edge_attr], dim=-1)) / 2
        x = x[edge_index.T].reshape(-1, 2 * self.n_hidden).relu()
        x = torch.cat((x, edge_attr.view(-1, edge_attr.shape[1])), dim=1)
        return self.mlp(x)

N_HIDDEN = 64
N_GNN_LAYERS = 2
EDGE_DIM = sample_batch['node', 'to', 'node'].edge_attr.shape[1] - 1
NUM_FEATURES = sample_batch['node'].x.shape[1]

base_model = GINe(num_features=NUM_FEATURES, num_gnn_layers=N_GNN_LAYERS, n_classes=2, n_hidden=N_HIDDEN, edge_updates=True, edge_dim=EDGE_DIM, dropout=0.0, final_dropout=0.105)
# Convert to Heterogeneous Model for Reverse Message Passing
model = to_hetero(base_model, te_data.metadata(), aggr='mean')

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Heterogeneous GINe Model initialized with {total_params:,} trainable parameters")


Heterogeneous GINe Model initialized with 139,010 trainable parameters


## 5. Device and Optimizer Setup


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
LR = 0.00621  # Tuned Multi-GNN learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Training on device: {device}")


Training on device: cuda


## 6. Training Loop


In [14]:
import os
import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score

@torch.no_grad()
def evaluate_hetero(loader, model, device, threshold=0.5):
    model.eval()
    preds, ground_truths = [], []
    for batch in tqdm.tqdm(loader, desc='Evaluating', leave=False):
        seed_ids = batch._seed_ids
        mask = torch.isin(batch['node', 'to', 'node'].edge_attr[:, 0].detach().cpu(), seed_ids.cpu())
        batch['node', 'to', 'node'].edge_attr = batch['node', 'to', 'node'].edge_attr[:, 1:]
        batch['node', 'rev_to', 'node'].edge_attr = batch['node', 'rev_to', 'node'].edge_attr[:, 1:]
        batch = batch.to(device)
        mask = mask.to(device)
        
        out_dict = model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict)
        out = out_dict[('node', 'to', 'node')][mask]
        probs = torch.softmax(out, dim=-1)[:, 1]
        pred = (probs >= threshold).long()
        
        preds.append(pred.cpu())
        ground_truths.append(batch['node', 'to', 'node'].y[mask].cpu())
        
    pred = torch.cat(preds).numpy()
    ground_truth = torch.cat(ground_truths).numpy()
    return {
        'f1': f1_score(ground_truth, pred, zero_division=0),
        'precision': precision_score(ground_truth, pred, zero_division=0),
        'recall': recall_score(ground_truth, pred, zero_division=0)
    }

N_EPOCHS = 30
W_CE1 = 1.0
W_CE2 = 50.0
best_val_f1 = 0.0

LOCAL_DIR = '/home/shreyas-nalle/Desktop/Delusional/model'
KAGGLE_DIR = '/kaggle/working'
SAVE_DIR = LOCAL_DIR if os.path.isdir(LOCAL_DIR) else (KAGGLE_DIR if os.path.isdir(KAGGLE_DIR) else '.')
model_save_path = os.path.join(SAVE_DIR, 'best_model.pt')

loss_fn = torch.nn.CrossEntropyLoss(weight=torch.FloatTensor([W_CE1, W_CE2]).to(device))
print(f'Training for {N_EPOCHS} epochs with Balanced HeteroData (W_CE2={W_CE2})...')

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    total_loss = total_examples = 0
    preds, ground_truths = [], []
    for batch in tqdm.tqdm(tr_loader, desc=f'Epoch {epoch}/{N_EPOCHS}', leave=False):
        optimizer.zero_grad()
        seed_ids = batch._seed_ids
        mask = torch.isin(batch['node', 'to', 'node'].edge_attr[:, 0].detach().cpu(), seed_ids.cpu())
        
        batch['node', 'to', 'node'].edge_attr = batch['node', 'to', 'node'].edge_attr[:, 1:]
        batch['node', 'rev_to', 'node'].edge_attr = batch['node', 'rev_to', 'node'].edge_attr[:, 1:]
        
        batch = batch.to(device)
        mask = mask.to(device)
        
        out_dict = model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict)
        out = out_dict[('node', 'to', 'node')]
        
        pred = out[mask]
        ground_truth = batch['node', 'to', 'node'].y[mask]
        if pred.numel() == 0: continue
        
        preds.append(pred.argmax(dim=-1).detach().cpu())
        ground_truths.append(ground_truth.detach().cpu())
        loss = loss_fn(pred, ground_truth)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * pred.shape[0]
        total_examples += pred.shape[0]
        
    train_pred = torch.cat(preds).numpy()
    train_gt = torch.cat(ground_truths).numpy()
    train_f1 = f1_score(train_gt, train_pred, zero_division=0)
    avg_loss = total_loss / max(total_examples, 1)
    
    val_metrics = evaluate_hetero(val_loader, model, device, threshold=0.5)
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save(model.state_dict(), model_save_path)
    print(f'Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Train F1: {train_f1:.4f} | Val F1: {val_metrics["f1"]:.4f} | Val Prec: {val_metrics["precision"]:.4f} | Val Rec: {val_metrics["recall"]:.4f}')

print(f'Training complete. Best Val F1: {best_val_f1:.4f}')


Training for 30 epochs with Balanced HeteroData (W_CE2=50.0)...


Epoch 01 | Loss: 0.0537 | Train F1: 0.7780 | Val F1: 0.0077 | Val Prec: 0.0039 | Val Rec: 0.9319


Epoch 02 | Loss: 0.0379 | Train F1: 0.8474 | Val F1: 0.0036 | Val Prec: 0.0018 | Val Rec: 0.9814


Epoch 03 | Loss: 0.0356 | Train F1: 0.8542 | Val F1: 0.0091 | Val Prec: 0.0045 | Val Rec: 0.9133


Epoch 04 | Loss: 0.0358 | Train F1: 0.8530 | Val F1: 0.0096 | Val Prec: 0.0048 | Val Rec: 0.9690


...

Epoch 26 | Loss: 0.0347 | Train F1: 0.8557 | Val F1: 0.0091 | Val Prec: 0.0046 | Val Rec: 0.9628


Epoch 27 | Loss: 0.0339 | Train F1: 0.8591 | Val F1: 0.0092 | Val Prec: 0.0046 | Val Rec: 0.9319


Epoch 28 | Loss: 0.0349 | Train F1: 0.8545 | Val F1: 0.0094 | Val Prec: 0.0047 | Val Rec: 0.9845


Epoch 29 | Loss: 0.0342 | Train F1: 0.8570 | Val F1: 0.0097 | Val Prec: 0.0049 | Val Rec: 0.9845


Epoch 30 | Loss: 0.0329 | Train F1: 0.8632 | Val F1: 0.0092 | Val Prec: 0.0046 | Val Rec: 0.9659
Training complete. Best Val F1: 0.0097


## 7. Final Test Evaluation


In [15]:
import os, shutil, glob
import torch
LOCAL_PATH = "/home/shreyas-nalle/Desktop/Delusional/model/best_model.pt"
KAGGLE_PATH = "/kaggle/working/best_model.pt"
CWD_PATH = "best_model.pt"
for p in [LOCAL_PATH, KAGGLE_PATH, CWD_PATH]:
    if os.path.exists(p):
        model_load_path = p
        break
else:
    raise FileNotFoundError("best_model.pt not found!")

print(f"Loading best model from: {model_load_path}")
model.load_state_dict(torch.load(model_load_path, map_location=device, weights_only=True))
final_metrics = evaluate_hetero(te_loader, model, device)
print("Final Test Set Results:")
print(f"  F1 Score  : {final_metrics['f1']:.4f}")
print(f"  Precision : {final_metrics['precision']:.4f}")
print(f"  Recall    : {final_metrics['recall']:.4f}")


Loading best model from: /kaggle/working/best_model.pt


Final Test Set Results:
  F1 Score  : 0.4660
  Precision : 0.8988
  Recall    : 0.3145
